First figuring out how to install and implement the right Python environment, ensuring it's active, then installing all necessary projects in the workspace.

In [ ]:
conda create -n final-project python=3.11
conda activate final-project

Installed pandas for standard data manipulation. openpyxl helps pandas to interpret Excel files. nltk is for language interpretation since the dataset is primarily words. scikit-learn for machine learning i.e regression, clustering. matplotlib creates data visualizations e.g plots and the manipulation of these. seaborn works with matplotlib to provide statistics visuals. 

In [ ]:
pip install pandas openpyxl nltk scikit-learn seaborn matplotlib setuptools urllib3 ipython ipython_pygments_lexers jedi jupyter_client jupyter_core matplotlib-inline nest-asyncio parso pexpect platformdirs prompt_toolkit psutil ptyprocess pure_eval comm debugpy decorator executing idna ipykernel

Initial analysis of the dataset to verify columns and that its pulling through the right Excel data

In [ ]:
import pandas as pd 
import numpy as np

file_path = '../data/award_data.xlsx'
try:
    df_raw = pd.read_excel(file_path, engine='openpyxl')
    print("Excel file loaded successfully.")

    print("\n--- Verifying Columns and Data Types (df.info()) ---")
        # This provides a concise summary of the DataFrame including column names, non-null counts, and data types.
    df_raw.info()

    print("\n--- Verifying Data Content (df.head()) ---")
        # This displays the first few rows of the DataFrame to give a snapshot of the data.
    print(df_raw.head())

except FileNotFoundError:
    print(f" Error: File not found. Check path: {file_path}")
except Exception as e:
    print(f"An error occurred: {e}")

Now we are filtering the rows we want to keep, keeping only awards granted by NASA, removing any other agency awards unrelated to space projects.

Defining the raw vs processed data paths (eye roll)

In [29]:
import pandas as pd
import os
import numpy as np
RAW_FILE_PATH = '../data/award_data.xlsx'
PROCESSED_FILE_PATH = '../data/processed/award_data_filtered.csv'
AGENCY_NAME = "National Aeronautics and Space Administration"
cols_to_drop = [
    'Branch', 
    'Program', 
    'Agency Tracking Number', 
    'Contract', 
    'Solicitation Number', 
    'Solicitation Year', 
    'Solicitation Close Date', 
    'Proposal Receipt Date', 
    'Date of Notification',
    'UEI',
    'DUNS',
    'Duns',
    'HUBZone Owned',
    'Company Website',
    'Contact Name',
    'Contact Title',
    'Contact Phone',
    'Contact Email',
    'PI Name',
    'PI Title',
    'PI Phone',
    'PI Email',
    'RI Name',
    'RI POC Name',
    'RI POC Phone',
]

try:
    # 1. Load Data
    df = pd.read_excel(RAW_FILE_PATH, engine='openpyxl')

    # 2. Standardize Column Names
    df.columns = df.columns.str.lower()
    
    # Update the drop list based on standardized names
    cols_to_drop = [col.lower() for col in cols_to_drop] 
    
    # 3. Filter by Agency
    rows_before_agency_filter = len(df)
    df_nasa = df[df['agency'] == AGENCY_NAME].copy()
    rows_after_agency_filter = len(df_nasa)
    print("\n--- Agency Filtering Report ---")
    print(f"Total rows before filtering: {rows_before_agency_filter}")
    print(f"Total rows kept for '{AGENCY_NAME}': {rows_after_agency_filter}")
    print(f"Percentage of data kept: {rows_after_agency_filter / rows_before_agency_filter:.2%}")
    
    # 4. Drop Irrelevant Columns
    rows_before_drop, cols_before_drop = df_nasa.shape
    df_nasa.drop(columns=cols_to_drop, inplace=True, errors='ignore')
    rows_after_drop, cols_after_drop = df_nasa.shape
    
    print("\n--- Column Dropping Report ---")
    print(f"DataFrame shape BEFORE drop: ({rows_before_drop}, {cols_before_drop})")
    print(f"Successfully dropped {cols_before_drop - cols_after_drop} columns.")
    print(f"DataFrame shape AFTER drop: ({rows_after_drop}, {cols_after_drop})")
    
    print("\nRemaining columns:")
    print(df_nasa.columns.tolist())

    print(f"\nData cleaning and filtering successful.")
    print(f"\nFinal data shape after filtering and dropping columns: {df_nasa.shape}")

    # 5. Save Cleaned Data
    os.makedirs(os.path.dirname(PROCESSED_FILE_PATH), exist_ok=True)
    df_nasa.to_csv(PROCESSED_FILE_PATH, index=False)
    
    print(f"\n Cleaned data saved to: {PROCESSED_FILE_PATH}")

except FileNotFoundError:
    print(f" Error: Raw file not found at {RAW_FILE_PATH}. Check your path.")



--- Agency Filtering Report ---
Total rows before filtering: 214381
Total rows kept for 'National Aeronautics and Space Administration': 19034
Percentage of data kept: 8.88%

--- Column Dropping Report ---
DataFrame shape BEFORE drop: (19034, 42)
Successfully dropped 24 columns.
DataFrame shape AFTER drop: (19034, 18)

Remaining columns:
['company', 'award title', 'agency', 'phase', 'proposal award date', 'contract end date', 'topic code', 'award year', 'award amount', 'socially and economically disadvantaged', 'woman owned', 'number employees', 'address1', 'address2', 'city', 'state', 'zip', 'abstract']

Data cleaning and filtering successful.

Final data shape after filtering and dropping columns: (19034, 18)

 Cleaned data saved to: ../data/processed/award_data_filtered.csv


CHECK BELOW - THIS IS WHERE YOU STOPPED

In [ ]:
null_counts = df.isnull().sum()
print(null_counts.sort_values(ascending=False))

cols_to_drop = ['Contact Title', 'RI POC Phone', 'RI Name', 'Solicitation Number',]
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

rows_before = len(df)
df.dropna(subset=['Abstract', 'Award Amount'], inplace=True)
rows_after = len(df)
print(f"Removed {rows_before - rows_after} rows due to missing critical data.")
print(f"The new total row count is {rows_after}.")

print("\n--- Verification of Missing Values ---")
print(df.isnull().sum().sort_values(ascending=False).head(5))
print(df.shape)